# E791 fit with efficiency and background

Fit the E791 $D^+\to\pi^-\pi^+\pi^+$ model with nonuniform efficiency, background, deliberately displaced start values, and fit-fraction closure.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import DecayChannel, DecayModel, Minimizer, NonResonant, Parameter, PhaseSpaceSample, RealImag, Resonance, enable_x64, weighted_resample
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency
enable_x64()


## 1. Model


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
fit2_polar = {"sigma": (1.17,205.7), "rho770": (1.0,0.0), "NR": (0.48,57.3), "f0_980": (0.43,165.0), "f2_1270": (0.76,57.3), "f0_1370": (0.26,105.4), "rho1450": (0.14,319.1)}
def internal_xy(name):
    mag, ph = fit2_polar[name]
    if name == "NR": ph += 180.0
    ph = np.deg2rad(ph)
    return mag*np.cos(ph), mag*np.sin(ph)
truth = {}
def coefficient(name, fixed=False):
    x,y = internal_xy(name)
    if fixed: return RealImag(x,y)
    truth[f"{name}.x"], truth[f"{name}.y"] = float(x), float(y)
    return RealImag(Parameter.coefficient(f"{name}.x", x, owner=name, step=0.01), Parameter.coefficient(f"{name}.y", y, owner=name, step=0.01))
c = {n: coefficient(n, fixed=(n=="rho770")) for n in fit2_polar}
components = [
 Resonance("sigma",(0,1),c["sigma"],mass=0.4780,width=0.3240,spin=0,resonance_radius=3.0,parent_radius=3.0),
 Resonance("rho770",(0,1),c["rho770"],mass=0.7693,width=0.1502,spin=1,resonance_radius=3.0,parent_radius=3.0),
 Resonance("f0_980",(0,1),c["f0_980"],mass=0.9750,width=0.0440,spin=0,resonance_radius=3.0,parent_radius=3.0),
 Resonance("f2_1270",(0,1),c["f2_1270"],mass=1.2750,width=0.1850,spin=2,resonance_radius=3.0,parent_radius=3.0),
 Resonance("f0_1370",(0,1),c["f0_1370"],mass=1.4340,width=0.1730,spin=0,resonance_radius=3.0,parent_radius=3.0),
 Resonance("rho1450",(0,1),c["rho1450"],mass=1.4650,width=0.3100,spin=1,resonance_radius=3.0,parent_radius=3.0),
 NonResonant(c["NR"])]
model = DecayModel(channel, components, normalization_method="gauss-legendre", normalization_order_m13=180, normalization_order_m23=180)
norm = model.normalization_sample
print("Generated fit fractions:")
model.print_fit_fractions(truth, normalization_sample=norm, include_interference=True)


## 2. Efficiency and background


In [ ]:
s12_min=(channel.daughter_masses[0]+channel.daughter_masses[1])**2; s12_max=(channel.parent_mass-channel.daughter_masses[2])**2
s13_min=(channel.daughter_masses[0]+channel.daughter_masses[2])**2; s13_max=(channel.parent_mass-channel.daughter_masses[1])**2
def scaled(data,key,low,high): return jnp.clip((data[key]-low)/(high-low),0.0,1.0)
efficiency = FunctionalEfficiency(lambda d: 0.55+0.30*scaled(d,"s12",s12_min,s12_max)+0.10*jnp.cos(jnp.pi*scaled(d,"s13",s13_min,s13_max)))
background = FunctionalBackground(lambda d: 0.50+1.20*scaled(d,"s12",s12_min,s12_max)+0.40*scaled(d,"s13",s13_min,s13_max))
eff_norm=efficiency(norm.as_dict()); bkg_norm=jnp.mean(norm.weights*background(norm.as_dict()))


## 3. Generate pseudo-data


In [ ]:
N_POOL=250_000; N_DATA=40_000; BACKGROUND_FRACTION_TRUE=0.18
pool=model.generate_phase_space(N_POOL,seed=2028); pool_cache=model.prepare_cache(pool,norm)
signal_weights=pool.weights*efficiency(pool.as_dict())*pool_cache.intensity(truth); background_weights=pool.weights*background(pool.as_dict())
n_background=int(round(N_DATA*BACKGROUND_FRACTION_TRUE)); n_signal=N_DATA-n_background
signal_data=weighted_resample(jax.random.key(2029),pool,signal_weights,n_signal,replace=True); background_data=weighted_resample(jax.random.key(2030),pool,background_weights,n_background,replace=True)
def merge(a,b):
    def joined(name):
        x,y=getattr(a,name),getattr(b,name); return None if x is None else jnp.concatenate((x,y))
    return PhaseSpaceSample(s12=joined("s12"),s13=joined("s13"),s23=joined("s23"),weights=jnp.ones((a.size+b.size,)),p1=joined("p1"),p2=joined("p2"),p3=joined("p3"))
data=merge(signal_data,background_data)


## 4. Fit


In [ ]:
cache=model.prepare_cache(data,norm,efficiency_normalization=eff_norm); eff_data=efficiency(data.as_dict()); bkg_data=background(data.as_dict())/bkg_norm
background_fraction=Parameter("background_fraction",0.12,bounds=(0.001,0.50),step=0.01); fit_parameters=(*model.parameters,background_fraction)
def nll(values):
    signal_pdf=eff_data*cache.intensity(values)/cache.normalization(values); f=values["background_fraction"]; total_pdf=(1-f)*signal_pdf+f*bkg_data
    return -jnp.sum(jnp.log(jnp.clip(total_pdf,min=1e-300)))
rng=np.random.default_rng(314159)
start={p.name:truth[p.name]+rng.normal(0.0,0.12) for p in model.parameters if not p.fixed}
start["background_fraction"]=float(np.clip(BACKGROUND_FRACTION_TRUE+rng.normal(0.0,0.04),0.01,0.49))
result=Minimizer(nll,fit_parameters,verbose=1).fit(start_values=start,simplex=True,ncall=40_000)
fit_values={p.name:float(result.values[p.name]) for p in model.parameters if not p.fixed}
print("valid:",result.valid,"NLL:",result.fval,"EDM:",result.fmin.edm)
print(f"{'parameter':20s} {'generated':>11s} {'start':>11s} {'fitted':>11s} {'error':>11s} {'pull':>9s}")
for p in model.parameters:
    if p.fixed: continue
    fit=float(result.values[p.name]); err=float(result.errors[p.name]); print(f"{p.name:20s} {truth[p.name]:11.5f} {start[p.name]:11.5f} {fit:11.5f} {err:11.5f} {(fit-truth[p.name])/err:9.3f}")
bf=float(result.values["background_fraction"]); be=float(result.errors["background_fraction"]); print(f"{'background_fraction':20s} {BACKGROUND_FRACTION_TRUE:11.5f} {start['background_fraction']:11.5f} {bf:11.5f} {be:11.5f} {(bf-BACKGROUND_FRACTION_TRUE)/be:9.3f}")
print("\nFitted fit fractions (without efficiency):"); model.print_fit_fractions(fit_values,normalization_sample=norm,include_interference=True)
print("\nFitted fit fractions (with efficiency):"); model.print_fit_fractions(fit_values,normalization_sample=norm,efficiency=efficiency,include_interference=True)


## 5. Projections


In [ ]:
projection_cache=model.prepare_cache(pool,norm,efficiency_normalization=eff_norm); efficiency_pool=efficiency(pool.as_dict()); background_pool=background(pool.as_dict())/bkg_norm
def proj(values,f,variable,bins):
    signal=pool.weights*efficiency_pool*projection_cache.intensity(values)/projection_cache.normalization(values); mix=(1-f)*signal+f*pool.weights*background_pool
    return np.histogram(np.asarray(getattr(pool,variable)),bins=bins,weights=np.asarray(mix))[0]
fig,axes=plt.subplots(1,2,figsize=(13,4.8),constrained_layout=True)
for ax,var,label in zip(axes,("s12","s13"),(r"$s_{12}$ [GeV$^2$]",r"$s_{13}$ [GeV$^2$]")):
    obs=np.asarray(getattr(data,var)); bins=np.linspace(obs.min(),obs.max(),70); centers=0.5*(bins[:-1]+bins[1:]); dh=np.histogram(obs,bins=bins)[0]
    gh=proj(truth,BACKGROUND_FRACTION_TRUE,var,bins); fh=proj(fit_values,bf,var,bins); gh*=dh.sum()/gh.sum(); fh*=dh.sum()/fh.sum()
    ax.errorbar(centers,dh,yerr=np.sqrt(np.maximum(dh,1)),fmt=".",label="toy data"); ax.step(centers,gh,where="mid",linestyle="--",label="generated model"); ax.step(centers,fh,where="mid",label="fitted model"); ax.set(xlabel=label,ylabel="events / bin"); ax.legend()
plt.show()
